<a href="https://colab.research.google.com/github/tavishgrade/nlp-language-model-portfolio/blob/main/Talktorial_7_Information_Extraction_TavishGrade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Talktorial 7 — Information Extraction: From Sentences to a Graph

**Graduate NLP · Information Extraction (Lecture 7) · Exposure #2** (lecture → **talktorial** → peer discussion)

In the lecture we went from one label per text to labels inside the text: which words name a drug or a
condition, which real thing each name refers to, and how the things are related. In this talktorial you do each
step yourself, on the lecture's own sentences:

- Part 1: a news-trained NER model (spaCy) on the United Airlines passage, written out as BIO tags
- Part 2: the same model on clinical sentences, then a biomedical model, scored with exact and partial match
- Part 3: a small language model asked for the entities, and a check of what it returns against the text
- Part 4: link the mentions to concepts, extract relations, and put them in a graph you can query

> ⚠️ **Colab users: do File → Save a copy in Drive first.** Part 2 downloads a biomedical model (about 250 MB)
> and Part 3 downloads Qwen (about 1 GB). Each takes a minute or two the first time. Everything runs on CPU; you
> do not need a GPU.

**By the end you will be able to:**
1. Turn a model's entities into BIO tags, and read a model's label set
2. Score NER output with precision, recall and F1, under exact and under partial match
3. Check a language model's extraction against the text it came from
4. Link mentions to concepts and build a small knowledge graph from relation triples

**Submit:** answer the prediction questions below, run all cells (outputs saved), complete the
**AI Verification Log**, answer the **reflection questions** in the *Before you submit* cell, then upload the
`.ipynb` to Canvas by **Sun 11:59 PM**.

## Before you code: make some predictions

Type your answers in this cell (double-click to edit). **One or two sentences each, and every answer
needs a reason.** Predictions are graded for being real, not for being right.

- **Q1. spaCy's small English model was trained on news and web text. Give it *"Lisinopril is used to treat
  hypertension."* What will it do with Lisinopril: miss it, or label it, and if so with which label? Why?**
  > *Your answer:* I think spaCy's news-trained model will probably miss Lisinopril entirely or label it incorrectly rather than recognize it as a drug. The lecture emphasized that NER models learn the entity types they were trained on, and spaCy's standard model is trained on news/web text with labels like person, organization, and location rather than drugs and diseases.
- **Q2. You ask a half-billion-parameter language model to list the drugs and problems in twelve short clinical
  sentences. Out of every ten strings it returns, how many will appear word for word in the sentence? Why?**
  > *Your answer:* My guess is about 8 out of 10 returned strings will appear word-for-word in the original sentence. The lecture pointed out that prompted LLMs sometimes paraphrase or normalize entities instead of copying the exact text span, so I expect a few outputs that are conceptually correct but not exact matches to the sentence.
- **Q3. "Which drugs used for hypertension can cause a cough?" One sentence says Accupril is prescribed for high
  blood pressure; another says quinapril may cause a dry cough. If you put the triples in a graph exactly as
  written, without linking, will the graph answer quinapril? Why or why not?**
  > *Your answer:* I do not think the graph would answer quinapril if the triples were added exactly as written without linking. One sentence contains Accupril and the other contains quinapril, and without entity linking the graph would treat them as different nodes, so the connection between hypertension and cough would be broken. The lecture's graph example specifically showed that linking Accupril and quinapril to the same concept is what makes the query work.

## Part 0 — Setup

Provided. Imports what we use (safe to re-run). `spacy` (with its small English model), `torch`, `transformers` and
`networkx` are preinstalled on Colab; the cell installs anything that is missing.

> **Colab:** if an install finishes with a message asking you to **restart the runtime**, do that
> (**Runtime → Restart session**) and run this cell again. That is normal, not an error.

In [ ]:
import json, os, re, time
import importlib.util, subprocess, sys
for pkg in ("spacy", "torch", "transformers", "networkx"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

import torch
import networkx as nx
import transformers
transformers.logging.set_verbosity_error()     # hide the download/load chatter; errors still show
torch.manual_seed(0)
print("spacy", spacy.__version__, "· en_core_web_sm", nlp.meta["version"], "· networkx", nx.__version__, "ready")

spacy 3.8.18 · en_core_web_sm 3.8.0 · networkx 3.7 ready


## Part 1 — NER on news, as BIO tags

This is the United Airlines passage from the lecture. spaCy's `en_core_web_sm` runs a tagger, a parser and an
entity recognizer over it. The entities come back as `doc.ents`: each one is a span of tokens with a start token
index `.start`, an end index `.end` (one past the last token, like a Python slice) and a label `.label_`.

In [ ]:
UNITED = ("Citing high fuel prices, United Airlines said Friday it has increased fares by $6 per round trip on "
          "flights to some cities also served by lower-cost carriers. American Airlines, a unit of AMR Corp., "
          "immediately matched the move, spokesman Tim Wagner said. United, a unit of UAL Corp., said the increase "
          "took effect Thursday and applies to most routes where it competes against discount carriers, such as "
          "Chicago to Dallas and Denver to San Francisco.")
doc = nlp(UNITED)
for ent in doc.ents:
    print(f"{ent.label_:8s} {ent.text!r:20s} tokens {ent.start}-{ent.end}")
print(len(doc.ents), "entities")

ORG      'United Airlines'    tokens 5-7
DATE     'Friday'             tokens 8-9
MONEY    '6'                  tokens 15-16
ORG      'American Airlines'  tokens 32-34
ORG      'AMR Corp.'          tokens 38-40
PERSON   'Tim Wagner'         tokens 47-49
ORG      'United'             tokens 51-52
ORG      'UAL Corp.'          tokens 56-58
DATE     'Thursday'           tokens 64-65
GPE      'Chicago'            tokens 79-80
GPE      'Dallas'             tokens 81-82
GPE      'Denver'             tokens 83-84
GPE      'San Francisco'      tokens 85-87
13 entities


The lecture showed the same entities as tags, one per token. Write `to_bio(doc)`, which returns one
`(token text, tag)` pair per token. The tag is `B-LABEL` on the first token of an entity, `I-LABEL` on the
tokens after it, and `O` on every token outside an entity. Build it from `doc.ents` and their `.start` / `.end`
token positions.

In [ ]:
# TODO: write to_bio(doc) -> a list of (token text, tag) pairs, one per token of doc.
#   Start every token at "O". For each ent in doc.ents: the token at ent.start gets "B-" + ent.label_,
#   and the tokens from ent.start + 1 up to (not including) ent.end get "I-" + ent.label_.
# Then print the pairs for the lecture's sentence:
#   "American Airlines, a unit of AMR Corp., immediately matched the move, spokesman Tim Wagner said."
def to_bio(doc):

    # start by assuming every token is outside an entity
    tags = ["O"] * len(doc)

    # fill in BIO tags using the entity spans from spaCy
    for ent in doc.ents:

        # first token in the entity
        tags[ent.start] = "B-" + ent.label_

        # remaining tokens in the entity
        for i in range(ent.start + 1, ent.end):
            tags[i] = "I-" + ent.label_

    # pair each token with its tag
    return [(token.text, tags[i]) for i, token in enumerate(doc)]


# lecture sentence
sent = nlp(
    "American Airlines, a unit of AMR Corp., immediately matched the move, spokesman Tim Wagner said."
)

for token, tag in to_bio(sent):
    print(f"{token:12s} {tag}")


American     B-ORG
Airlines     I-ORG
,            O
a            O
unit         O
of           O
AMR          B-ORG
Corp.        I-ORG
,            O
immediately  O
matched      O
the          O
move         O
,            O
spokesman    O
Tim          B-PERSON
Wagner       I-PERSON
said         O
.            O


In [ ]:
# Provided: does your function meet its contract? (If this fails, fix the cell above.)
_t = to_bio(nlp("American Airlines, a unit of AMR Corp., immediately matched the move."))
assert isinstance(_t, list) and all(len(p) == 2 for p in _t), "to_bio should return a list of (token, tag) pairs"
assert [tag for _, tag in _t][:3] == ["B-ORG", "I-ORG", "O"], f"American Airlines , -> B-ORG I-ORG O, got {_t[:3]}"
assert len(to_bio(doc)) == len(doc), "one pair per token"
print("Part 1 OK —", sum(tag.startswith("B-") for _, tag in to_bio(doc)), "entities begin in the United passage")

Part 1 OK — 13 entities begin in the United passage


**The lecture's slide counted 13 mentions in this passage, with the labels ORG, LOC, TIME, PER and MONEY. Which
labels does spaCy use instead, and for which mentions? spaCy's model card says what data it was trained on: which
dataset is it, and how does that explain the label names?**
> *Your answer:* The lecture slides labeled the entities as ORG, LOC, TIME, PER, and MONEY, but spaCy used ORG, GPE, DATE, PERSON, and MONEY on the same passage. For example, Friday and Thursday were labeled DATE instead of TIME, Tim Wagner was labeled PERSON instead of PER, and Chicago, Dallas, Denver, and San Francisco were labeled GPE (geopolitical entity) instead of LOC. The organization and money labels were essentially the same as the lecture's examples.

> According to spaCy's en_core_web_sm model documentation, the model was trained on the OntoNotes 5 corpus. OntoNotes uses a different label inventory than the simplified labels shown in the lecture, which explains why spaCy uses labels such as GPE and DATE instead of LOC and TIME. The model is designed for general news and web text, so its entity types follow the OntoNotes annotation scheme rather than the lecture's example labels.

### ✅ Verification checkpoint #1
Ask your AI assistant: *"Can spaCy's en_core_web_sm recognize drug names in clinical notes?"* Check its answer
against spaCy's model card for `en_core_web_sm` (what it was trained on, and its list of labels) and against your
own Part 2 table below. Record it in Part 5.

## Part 2 — The same model on clinical text

Twelve short clinical sentences, several from the lecture. Each one is written with its gold entities in
brackets, `[DRUG ...]` or `[PROBLEM ...]` (a disease, a symptom or a side effect). The cell below strips the
brackets and records each gold entity as a character span `(start, end, type)`, so `text[start:end]` is the
entity's text.

In [ ]:
CLINICAL = [
    "The doctor prescribed him 325 mg [DRUG Aspirin] p.o. 4x/day for 2 weeks.",
    "The doctor prescribed him [DRUG aspirin] for his [PROBLEM headache].",
    "The [DRUG Accupril] prescribed for his [PROBLEM high blood pressure] gave him a [PROBLEM headache].",
    "[DRUG Quinapril] may cause a [PROBLEM dry cough].",
    "She was started on [DRUG metformin] for [PROBLEM type 2 diabetes].",
    "[DRUG Lisinopril] was stopped after she developed a [PROBLEM persistent cough].",
    "[DRUG Lisinopril] is used to treat [PROBLEM hypertension].",
    "Patient took [DRUG Tylenol] for [PROBLEM back pain] with no relief.",
    "[DRUG Atorvastatin] 40 mg daily was prescribed for [PROBLEM high cholesterol].",
    "He reports [PROBLEM nausea] since starting [DRUG amoxicillin].",
    "[DRUG Warfarin] was held because of [PROBLEM bleeding gums].",
    "Given [DRUG MS] 4 mg IV for [PROBLEM pain].",
]

def parse(marked):
    """'[DRUG Aspirin] works' -> ('Aspirin works', [(0, 7, 'DRUG')])"""
    text, spans = "", []
    for m in re.finditer(r"\[(DRUG|PROBLEM) ([^\]]+)\]|[^\[]+", marked):
        if m.group(1):
            spans.append((len(text), len(text) + len(m.group(2)), m.group(1)))
            text += m.group(2)
        else:
            text += m.group(0)
    return text, spans

DOCS = [parse(s) for s in CLINICAL]
TEXTS = [t for t, _ in DOCS]
GOLD = [g for _, g in DOCS]
print(sum(len(g) for g in GOLD), "gold entities in", len(TEXTS), "sentences")

print("\nWhat the news model does with each gold entity:")
for text, gold in DOCS:
    ents = nlp(text).ents
    for s, e, typ in gold:
        hit = [f"{x.text!r} {x.label_}" for x in ents if x.start_char < e and s < x.end_char]
        print(f"  {typ:8s} {text[s:e]!r:22s} -> {', '.join(hit) if hit else 'missed'}")

24 gold entities in 12 sentences

What the news model does with each gold entity:
  DRUG     'Aspirin'              -> missed
  DRUG     'aspirin'              -> missed
  PROBLEM  'headache'             -> missed
  DRUG     'Accupril'             -> missed
  PROBLEM  'high blood pressure'  -> missed
  PROBLEM  'headache'             -> missed
  DRUG     'Quinapril'            -> 'Quinapril' ORG
  PROBLEM  'dry cough'            -> missed
  DRUG     'metformin'            -> missed
  PROBLEM  'type 2 diabetes'      -> '2' CARDINAL
  DRUG     'Lisinopril'           -> 'Lisinopril' PERSON
  PROBLEM  'persistent cough'     -> missed
  DRUG     'Lisinopril'           -> 'Lisinopril' PERSON
  PROBLEM  'hypertension'         -> missed
  DRUG     'Tylenol'              -> 'Tylenol' ORG
  PROBLEM  'back pain'            -> missed
  DRUG     'Atorvastatin'         -> 'Atorvastatin 40 mg' PERSON
  PROBLEM  'high cholesterol'     -> missed
  PROBLEM  'nausea'               -> missed
  DRUG     'a

spaCy has no DRUG or PROBLEM label, so it cannot score above zero here whatever it finds. The next cell
runs a biomedical NER model instead: `d4data/biomedical-ner-all`, a DistilBERT model fine-tuned on case reports.
It has more than 40 entity types; we keep the three that match our types: `Medication` becomes DRUG, and `Disease_disorder` and
`Sign_symptom` become PROBLEM.

In [ ]:
from transformers import pipeline
bio_ner = pipeline("token-classification", model="d4data/biomedical-ner-all", aggregation_strategy="first")   # "first": one label per whole word
TO_TYPE = {"Medication": "DRUG", "Disease_disorder": "PROBLEM", "Sign_symptom": "PROBLEM"}

def bio_entities(text):
    """The biomedical model's entities as (start, end, type) character spans, our two types only."""
    return [(r["start"], r["end"], TO_TYPE[r["entity_group"]]) for r in bio_ner(text) if r["entity_group"] in TO_TYPE]

BIO_PRED = [bio_entities(t) for t in TEXTS]
for text, pred in zip(TEXTS, BIO_PRED):
    print(text, "\n   ", [(text[s:e], typ) for s, e, typ in pred])

config.json:   0%|          | 0.00/5.00k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  266MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

The doctor prescribed him 325 mg Aspirin p.o. 4x/day for 2 weeks. 
    [('Aspirin', 'DRUG')]
The doctor prescribed him aspirin for his headache. 
    [('aspirin', 'DRUG')]
The Accupril prescribed for his high blood pressure gave him a headache. 
    []
Quinapril may cause a dry cough. 
    [('Quinapril', 'DRUG')]
She was started on metformin for type 2 diabetes. 
    [('metformin', 'DRUG')]
Lisinopril was stopped after she developed a persistent cough. 
    [('Lisinopril', 'DRUG')]
Lisinopril is used to treat hypertension. 
    [('Lisinopril', 'DRUG')]
Patient took Tylenol for back pain with no relief. 
    [('Tylenol', 'DRUG'), ('pain', 'PROBLEM')]
Atorvastatin 40 mg daily was prescribed for high cholesterol. 
    [('Atorvastatin', 'DRUG')]
He reports nausea since starting amoxicillin. 
    [('nausea', 'PROBLEM')]
Warfarin was held because of bleeding gums. 
    [('Warfarin', 'DRUG'), ('bleeding', 'PROBLEM')]
Given MS 4 mg IV for pain. 
    []


Now score it. Write `score(gold, pred, partial=False)`. Both arguments are lists with one entry per sentence,
and each entry is a list of `(start, end, type)` spans. Count over all sentences:

- **Exact match:** a predicted span matches a gold span when start, end and type are all the same.
- **Partial match:** a predicted span matches a gold span when the types are the same and the two character
  ranges overlap (`ps < ge and gs < pe`).
- **Precision** = predicted spans that match some gold span ÷ all predicted spans.
  **Recall** = gold spans matched by some predicted span ÷ all gold spans. **F1** = 2PR / (P + R).
  When a denominator is 0, use 0.

Return `(precision, recall, f1)` as plain floats.

In [ ]:
# TODO: write score(gold, pred, partial=False) -> (precision, recall, f1), as described above.
#   gold and pred: one list of (start, end, type) spans per sentence. Count over all sentences.
# Then print the biomedical model's scores twice: score(GOLD, BIO_PRED) and score(GOLD, BIO_PRED, partial=True).
def score(gold, pred, partial=False):

    matched_pred = 0
    matched_gold = 0

    total_pred = sum(len(p) for p in pred)
    total_gold = sum(len(g) for g in gold)

    # go sentence by sentence
    for g_spans, p_spans in zip(gold, pred):

        # count predicted spans that match a gold span
        for ps, pe, pt in p_spans:

            for gs, ge, gt in g_spans:

                if partial:
                    # same type + overlapping spans
                    match = (pt == gt) and (ps < ge and gs < pe)
                else:
                    # exact start/end/type match
                    match = (ps, pe, pt) == (gs, ge, gt)

                if match:
                    matched_pred += 1
                    break

        # count gold spans that were found
        for gs, ge, gt in g_spans:

            for ps, pe, pt in p_spans:

                if partial:
                    match = (pt == gt) and (ps < ge and gs < pe)
                else:
                    match = (ps, pe, pt) == (gs, ge, gt)

                if match:
                    matched_gold += 1
                    break

    # precision = correct predictions / all predictions
    precision = matched_pred / total_pred if total_pred else 0

    # recall = found gold entities / all gold entities
    recall = matched_gold / total_gold if total_gold else 0

    # standard F1 formula
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0
    )

    return (precision, recall, f1)


print("Exact :", score(GOLD, BIO_PRED))
print("Partial:", score(GOLD, BIO_PRED, partial=True))


Exact : (0.8333333333333334, 0.4166666666666667, 0.5555555555555556)
Partial: (1.0, 0.5, 0.6666666666666666)


In [ ]:
# Provided: checks your scorer on the lecture's exact-versus-partial slide, then prints the table.
_text = "for his high blood pressure"
_gold = [[(8, 27, "PROBLEM")]]           # high blood pressure
_pred = [[(13, 27, "PROBLEM")]]          # blood pressure
assert _text[8:27] == "high blood pressure" and _text[13:27] == "blood pressure"
assert score(_gold, _pred) == (0.0, 0.0, 0.0), f"exact: no match, so P = R = F1 = 0; got {score(_gold, _pred)}"
assert score(_gold, _pred, partial=True) == (1.0, 1.0, 1.0), "partial: the spans overlap, so P = R = F1 = 1"
assert score(_gold, [[(13, 27, "DRUG")]], partial=True)[0] == 0.0, "partial match still needs the same type"
SCORES = {"biomedical": {"exact": score(GOLD, BIO_PRED), "partial": score(GOLD, BIO_PRED, partial=True)}}
print("biomedical model, one type at a time (P, R, F1):")
for typ in ("DRUG", "PROBLEM"):
    g = [[x for x in gs if x[2] == typ] for gs in GOLD]
    p = [[x for x in ps if x[2] == typ] for ps in BIO_PRED]
    ex, pa = score(g, p), score(g, p, partial=True)
    print(f"  {typ:8s} exact {ex[0]:.3f} {ex[1]:.3f} {ex[2]:.3f}   partial {pa[0]:.3f} {pa[1]:.3f} {pa[2]:.3f}")
print("Part 2 OK")

biomedical model, one type at a time (P, R, F1):
  DRUG     exact 1.000 0.750 0.857   partial 1.000 0.750 0.857
  PROBLEM  exact 0.333 0.083 0.133   partial 1.000 0.250 0.400
Part 2 OK


**Was Q1 right? Name two drugs the news model gave a non-drug label, and the labels. Then quote the
biomedical model's exact and partial F1, and its recall for DRUG and for PROBLEM. Find one gold entity it got only
partly right: what did it return?**
> *Your answer:* Q1 was mostly right. I predicted that the news-trained spaCy model would miss Lisinopril, and while it did recognize it, it labeled Lisinopril as PERSON rather than as a drug. Other drugs were also assigned non-drug labels, including Quinapril -> ORG and Warfarin -> GPE.

> The biomedical model performed much better. Its exact-match F1 was 0.556, while its partial-match F1 was 0.667. Its recall was much higher for drugs than for problems: DRUG recall = 0.750 and PROBLEM recall = 0.083 under exact matching. Under partial matching, DRUG recall remained 0.750 and PROBLEM recall increased to 0.250.

> One gold entity that the biomedical model only got partly right was "back pain." The gold entity was the full span back pain, but the model returned only "pain" as a PROBLEM. Another example was "bleeding gums," which the model reduced to "bleeding." These boundary errors count as wrong under exact matching but correct under partial matching because the predicted spans overlap with the gold spans.

## Part 3 — Asking a language model for the entities

This is the lecture's zero-shot NER prompt, with our two types. `ask_entities(text)` is provided: it sends the
prompt to `Qwen2.5-0.5B-Instruct` (the same small model as last week), with greedy decoding, starts the model's
reply with `[` so that it answers in JSON instead of explaining itself, and parses the JSON it writes back into a list of `{"text": ..., "type": ...}` dicts. If the reply is not valid JSON it returns an empty list.

```
Extract every drug and every medical problem (a disease, symptom or side effect) from the sentence.
Return only a JSON list of {"text": ..., "type": ...} where type is Drug or Problem.
Sentence: <the text>
```

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
QWEN = "Qwen/Qwen2.5-0.5B-Instruct"
qwen_tok = AutoTokenizer.from_pretrained(QWEN)
qwen = AutoModelForCausalLM.from_pretrained(QWEN).float().eval()   # .float(): newer transformers load bfloat16, slow on CPU
PROMPT = ("Extract every drug and every medical problem (a disease, symptom or side effect) from the sentence.\n"
          'Return only a JSON list of {{"text": ..., "type": ...}} where type is Drug or Problem.\n'
          "Sentence: {text}")

def ask_entities(text):
    """The model's entities for one sentence: a list of {"text": ..., "type": ...} dicts (empty if unparseable)."""
    messages = [{"role": "user", "content": PROMPT.format(text=text)}]
    prompt = qwen_tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False) + "["
    enc = qwen_tok(prompt, return_tensors="pt")
    with torch.no_grad():
        out = qwen.generate(**enc, max_new_tokens=80, do_sample=False)
    reply = "[" + qwen_tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"\[.*\]", reply, re.S)
    try:
        items = json.loads(m.group(0)) if m else []
    except json.JSONDecodeError:
        return []
    return [x for x in items if isinstance(x, dict) and isinstance(x.get("text"), str)]

t0 = time.time()
LLM_RAW = [ask_entities(t) for t in TEXTS]
print(f"{time.time() - t0:.0f} s for {len(TEXTS)} sentences")
for text, items in zip(TEXTS, LLM_RAW):
    print(text, "\n   ", [(x["text"], x.get("type")) for x in items])

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

113 s for 12 sentences
The doctor prescribed him 325 mg Aspirin p.o. 4x/day for 2 weeks. 
    [('Aspirin', 'Drug'), ('Aspirin', 'Problem')]
The doctor prescribed him aspirin for his headache. 
    [('The doctor prescribed him aspirin', 'Drug'), ('for his headache', 'Problem')]
The Accupril prescribed for his high blood pressure gave him a headache. 
    [('The Accupril', 'Drug'), ('high blood pressure', 'Problem')]
Quinapril may cause a dry cough. 
    [('Quinapril', 'Drug'), ('dry cough', 'Problem')]
She was started on metformin for type 2 diabetes. 
    [('She was started on metformin', 'Drug'), ('for type 2 diabetes', 'Problem')]
Lisinopril was stopped after she developed a persistent cough. 
    [('Lisinopril', 'Drug'), ('persistent cough', 'Problem')]
Lisinopril is used to treat hypertension. 
    [('Lisinopril', 'Drug'), ('hypertension', 'Problem')]
Patient took Tylenol for back pain with no relief. 
    [('Patient', 'Drug'), ('Tylenol', 'Medicine'), ('back pain', 'Problem')]
Ato

The model returns strings, not positions. Before its answer can be scored, every string has to be found again
in the sentence. Write `ground(text, items)`, which returns two lists:

- `spans`: a `(start, end, type)` span for every item whose text occurs in the sentence (ignore case; take the
  first occurrence). For the type, upper-case what the model wrote: "Drug" becomes `"DRUG"`. The model sometimes
  writes another type, such as "Medicine"; keep it as it is (upper-cased), and the scorer will count it wrong. If an
  item has no type, use `""`.
- `invented`: the text of every item that does **not** occur in the sentence.

In [ ]:
# TODO: write ground(text, items) -> (spans, invented), as described above.
#   text.lower().find(s.lower()) gives the first position of s in text, or -1 if it is not there.
# Then print ground("Quinapril may cause a dry cough.", ...) for these three items:
#   {"text": "quinapril", "type": "Drug"}, {"text": "cough", "type": "Problem"}, {"text": "ACE inhibitor", "type": "Drug"}
def ground(text, items):

    spans = []
    invented = []

    for item in items:

        # get the text the model returned
        entity_text = item.get("text", "")

        # find it in the original sentence (ignore case)
        start = text.lower().find(entity_text.lower())

        if start == -1:
            # model returned something not actually in the sentence
            invented.append(entity_text)
            continue

        end = start + len(entity_text)

        # keep whatever type the model gave us
        # Drug -> DRUG, Problem -> PROBLEM, Medicine -> MEDICINE
        ent_type = str(item.get("type", "")).upper()

        spans.append((start, end, ent_type))

    return spans, invented


# example from the instructions
print(
    ground(
        "Quinapril may cause a dry cough.",
        [
            {"text": "quinapril", "type": "Drug"},
            {"text": "cough", "type": "Problem"},
            {"text": "ACE inhibitor", "type": "Drug"}
        ]
    )
)

([(0, 9, 'DRUG'), (26, 31, 'PROBLEM')], ['ACE inhibitor'])


In [ ]:
# Provided: checks your function, grounds every answer, and scores it beside the biomedical model.
_s, _inv = ground("Quinapril may cause a dry cough.", [{"text": "quinapril", "type": "Drug"},
                                                      {"text": "ACE inhibitor", "type": "Drug"}])
assert _s == [(0, 9, "DRUG")] and _inv == ["ACE inhibitor"], f"got {_s}, {_inv}"
grounded = [ground(t, items) for t, items in zip(TEXTS, LLM_RAW)]
LLM_PRED = [s for s, _ in grounded]
INVENTED = [x for _, inv in grounded for x in inv]
n_items = sum(len(items) for items in LLM_RAW)
print(f"The model returned {n_items} strings; {len(INVENTED)} are not in the sentence: {INVENTED}")
SCORES["qwen_zero_shot"] = {"exact": score(GOLD, LLM_PRED), "partial": score(GOLD, LLM_PRED, partial=True)}
print(f"\n{'system':16s} {'exact P':>8} {'R':>6} {'F1':>6}   {'partial P':>9} {'R':>6} {'F1':>6}")
for name, s in SCORES.items():
    print(f"{name:16s} {s['exact'][0]:8.3f} {s['exact'][1]:6.3f} {s['exact'][2]:6.3f}   "
          f"{s['partial'][0]:9.3f} {s['partial'][1]:6.3f} {s['partial'][2]:6.3f}")
print("Part 3 OK")

The model returned 25 strings; 0 are not in the sentence: []

system            exact P      R     F1   partial P      R     F1
biomedical          0.833  0.417  0.556       1.000  0.500  0.667
qwen_zero_shot      0.560  0.583  0.571       0.760  0.792  0.776
Part 3 OK


**Was Q2 right? Quote how many strings the model returned and how many were not in the sentence. Then find
two of its answers that are in the sentence and still wrong, and say which of the lecture's four failure modes
each one is. Which system gains more from partial match, and what does that tell you about its boundaries?**
> *Your answer:* Q2 was right. I predicted that most of the strings returned by the language model would appear word-for-word in the sentence. The model returned 25 strings, and 0 of them were not found in the sentence, so there were no invented entities in this run.

> However, some answers were still wrong even though the text appeared in the sentence. For example:

"Patient" -> Drug in "Patient took Tylenol for back pain with no relief." This is a type drift error because the model found a real span but assigned the wrong entity type.

"nausea" -> Drug in "He reports nausea since starting amoxicillin." This is also type drift, since nausea is a symptom and should be labeled as a Problem.

Another example is "The doctor prescribed him aspirin" -> Drug, which includes extra words beyond the entity itself. This is a boundary error because the model chose a different span than the gold annotation.

> The system that gained more from partial matching was Qwen. Its F1 increased from 0.571 under exact matching to 0.776 under partial matching, a gain of 0.205. The biomedical model increased from 0.556 to 0.667, a smaller gain of 0.111. This suggests that the language model often identified roughly the right entity but struggled more with exact boundaries, while the biomedical NER model was generally more precise about where entities started and ended.

### ✅ Verification checkpoint #2
Ask your AI assistant: *"When a large language model extracts entities from a sentence, does it only return text
that is in the sentence?"* Check its answer against your own Part 3 output first, then against the lecture's
failure-modes slide. Your run is twelve sentences with one model and one prompt: say what it can and cannot show
about the claim. Record it in Part 5.

## Part 4 — Linking, relations and a graph

**Linking.** `VOCAB` below is a tiny vocabulary: for each name it lists the candidate concepts, the way the
lecture's linking slide did (candidates first, then context). `CONCEPT_TYPE` gives each concept's type. *MS* has
three candidates; *Accupril* is a brand name for quinapril.

Write `link(mention, etype)`:
1. Look up the mention, lower-cased and stripped, in `VOCAB`. A mention that is not there has no entry: return
   the lower-cased mention itself (it becomes its own node).
2. Keep the candidates whose type in `CONCEPT_TYPE` equals `etype` (the NER type is the context we use).
3. If exactly one is left, return it. Otherwise return the lower-cased mention.

In [ ]:
VOCAB = {
    "accupril": ["quinapril"], "quinapril": ["quinapril"], "lisinopril": ["lisinopril"],
    "tylenol": ["acetaminophen"], "apap": ["acetaminophen"], "paracetamol": ["acetaminophen"],
    "ms": ["multiple sclerosis", "mitral stenosis", "morphine sulfate"],
    "high blood pressure": ["hypertension"], "hypertension": ["hypertension"], "hbp": ["hypertension"],
    "dry cough": ["cough"], "persistent cough": ["cough"], "cough": ["cough"],
    "high cholesterol": ["hypercholesterolemia"],
}
CONCEPT_TYPE = {"quinapril": "DRUG", "lisinopril": "DRUG", "acetaminophen": "DRUG", "morphine sulfate": "DRUG",
                "multiple sclerosis": "PROBLEM", "mitral stenosis": "PROBLEM", "hypertension": "PROBLEM",
                "cough": "PROBLEM", "hypercholesterolemia": "PROBLEM"}
print(len(VOCAB), "names,", len(CONCEPT_TYPE), "concepts")

14 names, 9 concepts


In [ ]:
# TODO: write link(mention, etype) as described above.
# Then print link(m, t) for: ("Accupril", "DRUG"), ("MS", "DRUG"), ("MS", "PROBLEM"),
#                             ("persistent cough", "PROBLEM"), ("nausea", "PROBLEM")
def link(mention, etype):

    # normalize the mention
    m = mention.strip().lower()

    # not in the vocabulary -> becomes its own node
    if m not in VOCAB:
        return m

    # keep only candidates with the right type
    candidates = [
        c for c in VOCAB[m]
        if CONCEPT_TYPE.get(c) == etype
    ]

    # if exactly one candidate fits, use it
    if len(candidates) == 1:
        return candidates[0]

    # otherwise we can't decide
    return m


# examples from the instructions
for m, t in [
    ("Accupril", "DRUG"),
    ("MS", "DRUG"),
    ("MS", "PROBLEM"),
    ("persistent cough", "PROBLEM"),
    ("nausea", "PROBLEM")
]:
    print(m, "->", link(m, t))


Accupril -> quinapril
MS -> morphine sulfate
MS -> ms
persistent cough -> cough
nausea -> nausea


**Relations and the graph.** `relations(text, spans)` is provided. It is the heuristic approach from the
lecture: for each drug and problem in the same sentence, it reads the words between them. A cue such as *cause*,
*gave*, *developed*, *since starting* or *because of* makes the pair an ADE (adverse drug event); *for* or *treat*
makes it TREATS. The cell then builds a graph two ways: with every mention linked by your `link()`, and with the
mentions exactly as written. Each edge keeps the sentence it came from.

In [ ]:
ADE_CUES = ("cause", "gave", "developed", "since starting", "because of")
TREATS_CUES = ("for", "treat")

def relations(text, spans):
    """(drug mention, relation, problem mention) triples from one sentence, by cue words between the two."""
    drugs = [(s, e) for s, e, t in spans if t == "DRUG"]
    problems = [(s, e) for s, e, t in spans if t == "PROBLEM"]
    triples = []
    for ds, de in drugs:
        for ps, pe in problems:
            between = text[min(de, pe):max(ds, ps)].lower()
            words = re.findall(r"[a-z]+", between)
            if any(cue in between for cue in ADE_CUES):
                triples.append((text[ds:de], "ADE", text[ps:pe]))
            elif any(cue in words or (cue == "treat" and "treat" in between) for cue in TREATS_CUES):
                triples.append((text[ds:de], "TREATS", text[ps:pe]))
    return triples

def build_graph(texts, spans_per_text, linked=True):
    G = nx.MultiDiGraph()
    for text, spans in zip(texts, spans_per_text):
        for d, rel, p in relations(text, spans):
            a = link(d, "DRUG") if linked else d
            b = link(p, "PROBLEM") if linked else p
            G.add_edge(a, b, relation=rel, sentence=text)
    return G

def drugs_that(G, treats, causes):
    """Drugs with a TREATS edge to `treats` and an ADE edge to `causes`."""
    def targets(drug, rel):
        return {v for _, v, r in G.out_edges(drug, data="relation") if r == rel}
    return sorted(n for n in G if treats in targets(n, "TREATS") and causes in targets(n, "ADE"))

TRIPLES = [(t, r) for t, g in DOCS for r in relations(t, g)]
for text, (d, rel, p) in TRIPLES:
    print(f"  ({d}, {rel}, {p})")
G = build_graph(TEXTS, GOLD, linked=True)
G_raw = build_graph(TEXTS, GOLD, linked=False)
print(f"\nlinked graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges · "
      f"unlinked: {G_raw.number_of_nodes()} nodes, {G_raw.number_of_edges()} edges")
print("Which drugs used for hypertension can cause a cough?")
print("  linked:  ", drugs_that(G, "hypertension", "cough"))
print("  unlinked:", drugs_that(G_raw, "hypertension", "cough"))
for d in drugs_that(G, "hypertension", "cough"):
    for _, v, data in G.out_edges(d, data=True):
        if v in ("hypertension", "cough"):
            print(f"    {d} -{data['relation']}-> {v}   from: {data['sentence']}")

  (aspirin, TREATS, headache)
  (Accupril, TREATS, high blood pressure)
  (Accupril, ADE, headache)
  (Quinapril, ADE, dry cough)
  (metformin, TREATS, type 2 diabetes)
  (Lisinopril, ADE, persistent cough)
  (Lisinopril, TREATS, hypertension)
  (Tylenol, TREATS, back pain)
  (Atorvastatin, TREATS, high cholesterol)
  (amoxicillin, ADE, nausea)
  (Warfarin, ADE, bleeding gums)
  (MS, TREATS, pain)

linked graph: 18 nodes, 12 edges · unlinked: 21 nodes, 12 edges
Which drugs used for hypertension can cause a cough?
  linked:   ['lisinopril', 'quinapril']
  unlinked: []
    lisinopril -ADE-> cough   from: Lisinopril was stopped after she developed a persistent cough.
    lisinopril -TREATS-> hypertension   from: Lisinopril is used to treat hypertension.
    quinapril -TREATS-> hypertension   from: The Accupril prescribed for his high blood pressure gave him a headache.
    quinapril -ADE-> cough   from: Quinapril may cause a dry cough.


In [ ]:
# Provided: errors flow downstream. The same graph, built from the biomedical model's entities instead of the gold ones.
assert link("Accupril", "DRUG") == "quinapril" and link("MS", "DRUG") == "morphine sulfate", "link() is off"
assert link("MS", "PROBLEM") == "ms" and link("nausea", "PROBLEM") == "nausea", "link() is off for MS/PROBLEM or a NIL"
G_bio = build_graph(TEXTS, BIO_PRED, linked=True)
gold_edges = {(u, v, r) for u, v, r in G.edges(data="relation")}
bio_edges = {(u, v, r) for u, v, r in G_bio.edges(data="relation")}
print(f"edges from gold entities: {len(gold_edges)} · from the model's entities: {len(bio_edges)} · "
      f"in both: {len(gold_edges & bio_edges)}")
print("  missing from the model's graph:")
for e in sorted(gold_edges - bio_edges):
    print("    ", e)
print("  extra in the model's graph:")
for e in sorted(bio_edges - gold_edges):
    print("    ", e)
print("answer from the model's graph:", drugs_that(G_bio, "hypertension", "cough"))
print("Part 4 OK")

edges from gold entities: 12 · from the model's entities: 2 · in both: 0
  missing from the model's graph:
     ('acetaminophen', 'back pain', 'TREATS')
     ('amoxicillin', 'nausea', 'ADE')
     ('aspirin', 'headache', 'TREATS')
     ('atorvastatin', 'hypercholesterolemia', 'TREATS')
     ('lisinopril', 'cough', 'ADE')
     ('lisinopril', 'hypertension', 'TREATS')
     ('metformin', 'type 2 diabetes', 'TREATS')
     ('morphine sulfate', 'pain', 'TREATS')
     ('quinapril', 'cough', 'ADE')
     ('quinapril', 'headache', 'ADE')
     ('quinapril', 'hypertension', 'TREATS')
     ('warfarin', 'bleeding gums', 'ADE')
  extra in the model's graph:
     ('acetaminophen', 'pain', 'TREATS')
     ('warfarin', 'bleeding', 'ADE')
answer from the model's graph: []
Part 4 OK


**Was Q3 right? Quote both answers to the hypertension question, and for each drug in the linked answer name
the mentions that had to be linked for its path to exist. Then pick one edge missing from the model's graph and trace it back to the
entity mistake that caused it.**
> *Your answer:* Q3 was right. I predicted that the graph would not be able to answer the question without linking, and that is exactly what happened. For the question "Which drugs used for hypertension can cause a cough?", the linked graph returned ['lisinopril', 'quinapril'], while the unlinked graph returned[].

For quinapril, the path only exists because two pairs of mentions were linked together. "Accupril" was linked to the concept quinapril, and "high blood pressure" was linked to hypertension. Without those links, the TREATS relation and the ADE relation would connect to different nodes and the path would be broken.

For lisinopril, the path depends on linking "persistent cough" to the concept cough. One sentence says Lisinopril caused a persistent cough, while another says it treats hypertension. Linking allows both facts to connect to the same cough node.

One missing edge in the model's graph was: (quinapril, hypertension, TREATS)

Tracing it back, the biomedical model returned no entities at all for the sentence: "The Accupril prescribed for his high blood pressure gave him a headache."

Because it missed both the drug mention (Accupril) and the problem mention (high blood pressure), the relation extraction step never had the entities needed to create the TREATS edge. This is exactly the error-propagation problem discussed in the lecture: when NER misses an entity, the downstream relation and graph construction steps cannot recover it.

## Part 5 — AI Verification Log

One block per checkpoint: the **question you asked**, the **claim** that came back, the **source** you
checked it against, your **verdict**, and **what you found when you checked** — what the source actually
said, or what your own output actually showed. Then add at least one claim you chose yourself.

**How the log is graded** — the same four rows as the assignment page:
- **The question.** A real question, and a follow-up where the first answer was thin. *"Isn't NER basically
  solved?"* is fishing for agreement; *"Where does NER still fail on clinical text?"* is a question.
- **Source quality.** The right kind for the kind of claim: the lecture, the textbook or the paper for a
  definition or a research finding, the library documentation or model card for how a model behaves and what it
  was trained on, **your own output** for a claim about what these models do on these sentences.
- **Thoroughness.** Every checkpoint, plus a claim you chose yourself.
- **What you found.** The finding says what the source *said* or what your output *showed*. Naming a
  caveat, a boundary condition, or a case where the claim breaks is what makes a block excellent
  rather than satisfactory. The AI does not have to be wrong.

*(Double-click this cell to edit. Type after each bold label, then Shift+Enter to render it.)*

**A worked example, filled in so you can see the bar.** This one is not yours to answer:

- **What you asked:** "In BIO tagging, what tag does a one-word entity get?"
- **What it told you:** A one-word entity gets the tag S, for single.
- **Source checked:** The lecture's BIOES and BIO slides, and my own Part 1 output
- **Verdict:** ⚠️ partly right
- **What you found when you checked:** S is a BIOES tag, not a BIO tag. The lecture's BIOES slide has S for a one-token entity and E for the last token of a longer one; the BIO slide has only B, I and O. In BIO a one-word entity is just B: in my Part 1 output "Friday" is B-DATE and the next token is O. So the answer is right for BIOES and wrong for the scheme I asked about.

**The same block, written the way that does not count.** Same question, same source:

- **What you found when you checked:** The lecture covers BIO tags. ✅

That does not say what the slides said about S, and it did not look at the output. A finding that could have been
written without opening the source is not a check.

---

### Claim 1 — checkpoint #1
- **What you asked:** "Can spaCy's en_core_web_sm recognize drug names in clinical notes?"
- **What it told you:** The model may recognize some drug names, but it was not trained for clinical named entity recognition and does not have a dedicated DRUG label. Clinical NER models such as scispaCy or Med7 are more appropriate for extracting medications from clinical text.
- **Source checked:** spaCy's en_core_web_sm model page; Lecture 7 slides on NER and domain-specific entity types; and my Part 1 prediction and output.
- **Verdict:** ⚠️ Partly right
- **What you found when you checked:** The AI correctly stated that en_core_web_sm is a general-domain model and does not include a dedicated DRUG entity type. This matches the lecture's discussion that NER systems learn the entity types they were trained on and that biomedical applications often require specialized types such as drugs, diseases, proteins, or chemicals. The lecture specifically noted that a model trained on one set of entity types will not automatically recognize the entity types of another domain.

> However, the AI missed an important caveat from the lecture. The issue is not only that spaCy lacks a DRUG label; it is also that the model was trained on news and web text, while clinical language contains many specialized terms and abbreviations. The lecture emphasized that biomedical text requires different entity categories and often uses specialized models such as BioBERT because domain vocabulary differs substantially from news text. In other words, even if spaCy recognizes a drug name as some generic entity, that does not mean it can reliably perform clinical NER. The lecture's examples showed that specialized biomedical models are needed when the target entities are drugs, diseases, doses, and other clinical concepts.

> The AI was therefore correct about the general limitation, but it focused on the label set and did not fully explain the domain-mismatch problem emphasized in the lecture.

### Claim 2 — checkpoint #2
- **What you asked:** "When a large language model extracts entities from a sentence, does it only return text that is in the sentence?"
- **What it told you:** No. An LLM can return entities that are not literally present in the text because it may normalize, infer, or generate related concepts. For example, it might extract "ibuprofen" from a sentence containing only "Advil."
- **Source checked:** Lecture 7 slides on zero-shot NER and LLM failure modes, especially the slides comparing BERT+CRF to prompted LLMs; and my own Part 3 output.
- **Verdict:** ⚠️ Partly right
- **What you found when you checked:** The lecture supports the AI's main claim. One of the lecture's listed failure modes for prompted LLMs is that they can return a span that is not actually in the text, such as returning "by mouth" when the source sentence contains only"p.o." The lecture also notes that relation-extraction prompts may generate normalized concepts such as "hypertension" even when the sentence only says "high blood pressure." These examples show that LLMs can generate entities that are not literal text spans.

> However, my own Part 3 results showed an important caveat. In this experiment, the language model returned 25 entity strings and 0 were invented. Every extracted string appeared somewhere in the original sentence. The model still made mistakes, but they were mostly boundary and type errors. For example, it returned"The doctor prescribed him aspirin " instead of just "aspirin," and labeled "nausea" as a Drug rather than a Problem. So the AI was correct that LLMs can generate entities that do not appear in the text, but that failure mode did not occur in my particular run. Instead, my results showed that incorrect boundaries and incorrect labels were the bigger source of error.

### Claim 3 — one you chose yourself
*A claim about what these models do is checked against your own output — a row of the Part 2 table, a string in
Part 3, or an edge in Part 4 belong here.*
- **What you asked:** "Does a higher NER F1 score always lead to better relation extraction and knowledge graph results?"
- **What it told you:** No. A higher NER F1 score can improve downstream performance, but it does not guarantee better relation extraction or knowledge graph results because relation extraction, entity linking, and graph construction introduce their own sources of error.
- **Source checked:** Lecture 7 slides on error propagation in information extraction pipelines, entity linking, relation extraction, and graph construction; and my own Part 2 and Part 4 results.
- **Verdict:** ⚠️ Partly right
- **What you found when you checked:** The AI's main claim matches the lecture. The lecture explicitly states that errors flow downstream in a pipeline: if an entity is missed, the correct relation cannot be extracted, and the graph cannot contain the correct edge. The lecture's example notes that if NER misses part of "high blood pressure," the relation extraction stage may be unable to produce the correct triple.

> However, my results showed an additional nuance. The biomedical model achieved an exact-match NER F1 of 0.556 and a partial-match F1 of 0.667, which sounds reasonably good. Despite that, the graph built from the model's entities contained only 2 edges, compared with 12 edges in the graph built from the gold entities. It also failed to answer the question "Which drugs used for hypertension can cause a cough?", returning[] instead of['lisinopril', 'quinapril']. In this case, missing a few key entities had a much larger impact on the graph than the overall NER F1 score suggested. My results support the AI's point that NER quality alone does not determine graph quality, but they also show that some entity errors matter far more than others because they break entire reasoning paths in the graph.

**Verdict key:** ✅ correct · ⚠️ partly right / needs caveat · ❌ wrong.

## Part 6 — Save your artifact (shared format → final project)

In [ ]:
# TODO: Build a dict `artifact` in the shared envelope
#   {"week": 10, "task": "information_extraction", "source": "...", "outputs": {...}}
# "week" is the week of the semester (this is week 10), not the talktorial number. "source" is one line saying what
# data and models you used (the clinical sentences, the biomedical model, Qwen).
# `outputs` is a DICT with named keys. Fill it from the variables in this notebook, not by typing values in:
#   "entities": for each sentence, its text and the gold, biomedical and Qwen spans (TEXTS, GOLD, BIO_PRED, LLM_PRED)
#   "scores":   SCORES (started in Part 2, finished in Part 3)
#   "invented": INVENTED
#   "triples":  one dict per edge of the linked graph G: {"head", "relation", "tail", "sentence"}
#               (for u, v, d in G.edges(data=True): d["relation"], d["sentence"])
#   "answer":   drugs_that(G, "hypertension", "cough")
# Save it with json.dump as talktorial7_artifact.json, then print it (you may leave entities out of the print).
artifact = {
    "week": 10,
    "task": "information_extraction",
    "source": "Clinical sentences with gold annotations, d4data/biomedical-ner-all, and Qwen2.5-0.5B-Instruct",
    "outputs": {

        # entities from all three sources
        "entities": [
            {
                "text": text,
                "gold": gold,
                "biomedical": bio,
                "qwen": llm
            }
            for text, gold, bio, llm in zip(TEXTS, GOLD, BIO_PRED, LLM_PRED)
        ],

        # Part 2 + Part 3 scores
        "scores": SCORES,

        # strings returned by Qwen that were not found in the sentence
        "invented": INVENTED,

        # graph edges
        "triples": [
            {
                "head": u,
                "relation": d["relation"],
                "tail": v,
                "sentence": d["sentence"]
            }
            for u, v, d in G.edges(data=True)
        ],

        # final graph query answer
        "answer": drugs_that(G, "hypertension", "cough")
    }
}

with open("talktorial7_artifact.json", "w") as f:
    json.dump(artifact, f, indent=2)

# print everything except the large entities section
print({
    "week": artifact["week"],
    "task": artifact["task"],
    "source": artifact["source"],
    "outputs": {
        k: v
        for k, v in artifact["outputs"].items()
        if k != "entities"
    }
})

{'week': 10, 'task': 'information_extraction', 'source': 'Clinical sentences with gold annotations, d4data/biomedical-ner-all, and Qwen2.5-0.5B-Instruct', 'outputs': {'scores': {'biomedical': {'exact': (0.8333333333333334, 0.4166666666666667, 0.5555555555555556), 'partial': (1.0, 0.5, 0.6666666666666666)}, 'qwen_zero_shot': {'exact': (0.56, 0.5833333333333334, 0.5714285714285714), 'partial': (0.76, 0.7916666666666666, 0.7755102040816326)}}, 'invented': [], 'triples': [{'head': 'aspirin', 'relation': 'TREATS', 'tail': 'headache', 'sentence': 'The doctor prescribed him aspirin for his headache.'}, {'head': 'quinapril', 'relation': 'TREATS', 'tail': 'hypertension', 'sentence': 'The Accupril prescribed for his high blood pressure gave him a headache.'}, {'head': 'quinapril', 'relation': 'ADE', 'tail': 'headache', 'sentence': 'The Accupril prescribed for his high blood pressure gave him a headache.'}, {'head': 'quinapril', 'relation': 'ADE', 'tail': 'cough', 'sentence': 'Quinapril may cause

In [ ]:
# Provided: checks the artifact is in the shared shape. If it fails, fix the cell above.
import json, os
assert os.path.exists("talktorial7_artifact.json"), "talktorial7_artifact.json was not written"
_a = json.load(open("talktorial7_artifact.json"))
_missing = {"week", "task", "source", "outputs"} - set(_a)
assert not _missing, f"envelope is missing {_missing}"
assert isinstance(_a["outputs"], dict) and _a["outputs"], "outputs must be a non-empty dict with named keys"
assert len(_a["outputs"].get("triples", [])) == G.number_of_edges(), "include one triple per edge of G"
assert {"head", "relation", "tail", "sentence"} <= set(_a["outputs"]["triples"][0]), "each triple needs head, relation, tail, sentence"
print("artifact OK — outputs keys:", sorted(_a["outputs"]))

artifact OK — outputs keys: ['answer', 'entities', 'invented', 'scores', 'triples']


## Before you submit — a few questions, honestly answered

These are not graded for being impressive. They are graded for being real, and they are
how I find out what is not landing while there is still time to do something about it.
*(Double-click this cell to type your answers, then Shift+Enter.)*

**1. What surprised you?**
*(Scroll back to your predictions at the top — where was your intuition wrong, and why?)*

> *Your answer:* I was surprised that the zero-shot language model ended up with a higher F1 score than the biomedical NER model, even though the biomedical model was trained for clinical text. At the same time, the language model made some strange mistakes, like labeling "nausea" as a drug or including extra words in entity spans. It was interesting to see that a model can perform reasonably well overall while still making obvious errors on individual examples.

**2. What got you stuck this week, and what did you do about it?**
*("I am still stuck on X" is a complete and fully-credited answer.)*

> *Your answer:* The part that took me the longest was understanding how the evaluation worked, especially the difference between exact match and partial match. At first I didn't understand why a prediction could be considered wrong under one metric and correct under another. Going back to the lecture example with "high blood pressure" versus "blood pressure" and then looking at my own outputs helped me understand how boundary errors affect precision, recall, and F1.

**3. What is one question you still have?**

> *Your answer:* If large language models can extract entities, perform entity linking, and extract relations all from a prompt, when is it still better to use a traditional pipeline with separate NER, linking, and relation extraction models?

---

### Working with AI

*There is no right answer here — "it frustrated me" and "I am not sure it helped" are as useful to me as anything positive. Answer honestly rather than helpfully.*

**a. Which AI did you use this week?** Name the assistant, and the model if you know it — e.g. "ChatGPT", "Claude", "Gemini", "Copilot", a local model, or "none this week", which is a perfectly good answer.

> *Your answer:* ChatGPT and Copilot.

**b. After completing this week’s AI-supported task and checking the AI’s response, what did you think and feel about your experience with AI?**

*You might reflect on things such as what surprised you, what confused or frustrated you, what helped you learn, what made you feel more or less confident, or whether checking the AI’s response changed your thinking or feelings.*

> *Your answer:* This week reinforced that AI is often directionally correct but can miss important details. In several of the verification checks, the main idea was right, but the lecture, model cards, or my own outputs revealed caveats that were easy to overlook. I found the verification process useful because it forced me to compare the AI's claims against actual evidence instead of accepting the answer at face value. It made me more confident using AI as a learning tool, but also more aware of the need to check where its answers come from and whether they fully match the task.



## Wrap-up

You ran the whole pipeline from the lecture. A news model found the entities in a news passage and fell apart on
clinical text: drugs came back as people, organizations and places. A biomedical model found most of the drugs
and few of the problems, and the exact and partial scores showed where its boundaries differ from the gold ones.
The language model returned strings that had to be found again in the text before they could be scored, and you
checked each one. Then linking turned Accupril and quinapril into one node, and a question no single sentence
answers became a path of two edges.

Next week is RAG: retrieving text to help a language model answer. The week after, GraphRAG retrieves from graphs
like the one you built.

**Optional extensions:** add a third type (Dose) to the gold spans and the prompt, and see which system handles
"325 mg"; try a few-shot version of the Part 3 prompt and count the invented strings again; build the graph from
the language model's grounded spans and compare its edges with the gold graph.

**Submit:** Run all cells, then **File → Download .ipynb** → upload to Canvas with the Verification
Log completed.

---

## Stuck for more than thirty minutes? Hints

Try without these first — working it out is where the points are. Using a hint costs nothing.

**Part 1, to_bio.** `tags = ["O"] * len(doc)`, then loop over `doc.ents` and set `tags[ent.start]` and
`tags[ent.start + 1 : ent.end]`. Return `[(tok.text, tag) for tok, tag in zip(doc, tags)]`.

**Part 2, score.** Write a small `match(p, g)` first. Then count the predicted spans for which
`any(match(p, g) for g in that sentence's gold)` is true, and the gold spans matched by any prediction.

**Part 3, ground.** `i = text.lower().find(x["text"].lower())`; the span is `(i, i + len(x["text"]), ...)`.

**Part 4, link.** `[c for c in VOCAB.get(key, []) if CONCEPT_TYPE.get(c) == etype]`, then check its length.